# Liverpool Walkability Analysis

This notebook explores walkability and street network connectivity in Liverpool using Python and OpenStreetMap data.

The aim is to analyse the structure of Liverpool's walking network and visualise connectivity patterns using static and interactive geospatial maps.

## 1. Import Libraries

The libraries below are used for retrieving OpenStreetMap data, processing geospatial information, analysing the street network, and creating maps.

In [ ]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import folium

## 2. Retrieve Liverpool Walking Network

This section retrieves the walking network for Liverpool using OpenStreetMap data through OSMnx. The network is represented as a graph, with intersections or endpoints as nodes and walking routes as edges.

In [ ]:
place_name = "Liverpool, England, United Kingdom"

G = ox.graph_from_place(place_name, network_type="walk")

## 3. Convert Network to GeoDataFrames

The walking network is converted into GeoDataFrames so that the nodes and edges can be analysed and mapped as spatial data.

In [ ]:
nodes, edges = ox.graph_to_gdfs(G)

## 4. Basic Network Summary

This section checks the size of the walking network by looking at the number of nodes and edges.

In [ ]:
# Calculate node degree (how many streets meet at each intersection)
# Higher degree = more route choices = better for walkers
node_degree = dict(G.degree())
nodes['degree'] = nodes.index.map(node_degree)

# Check the distribution of degree values
print("Node degree summary:")
print(nodes['degree'].describe())
print(f"\nNodes with degree 1 (dead ends): {(nodes['degree'] == 1).sum()}")
print(f"Nodes with degree 3+ (intersections): {(nodes['degree'] >= 3).sum()}")

In [ ]:
print(f"Nodes: {len(nodes)}, Edges: {len(edges)}")

## 5. Analyse Street Connectivity

Node degree is used as a simple measure of street connectivity. A higher node degree means more walking routes connect at that point, while a lower node degree suggests fewer connections.

In [ ]:
nodes["degree"] = nodes.index.map(dict(G.degree()))
nodes[["degree"]].head()

## 6. Static Map: Liverpool Walking Network

This static map shows the structure of Liverpool's walking network. It helps reveal the density and pattern of walkable streets across the city.

In [ ]:
# Static Map 1: Liverpool walking street network — geographic context
# This map shows the full extent of the pedestrian-accessible network across the city
fig, ax = plt.subplots(figsize=(11, 11))

# Plot edges (street segments) in a dark colour on a white background
edges.plot(ax=ax, linewidth=0.2, color="#2c2c2c", alpha=0.6)

# Title and subtitle
ax.set_title("Pedestrian Street Network — Liverpool, England",
             fontsize=17, fontweight="bold", pad=15)
ax.text(0.5, 0.97, "Walking-accessible edges derived from OpenStreetMap via OSMnx",
        transform=ax.transAxes, fontsize=9, ha="center", color="grey", style="italic")

# Data source caption bottom left
ax.text(0.01, 0.01, "Data: OpenStreetMap contributors (2024) | CRS: EPSG:4326",
        transform=ax.transAxes, fontsize=8, color="grey")

# North arrow (simple text annotation)
ax.annotate("N ↑", xy=(0.97, 0.05), xycoords="axes fraction",
            fontsize=13, ha="center", fontweight="bold", color="#333333")

ax.set_facecolor("white")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 7. Static Map: Street Connectivity

This map visualises node degree to show areas with higher and lower levels of street connectivity.

In [ ]:
# Static Map 2: Node degree map — connectivity across Liverpool
# Node degree = number of streets meeting at each intersection
# Higher values (warmer colours) = more route choices for pedestrians

fig, ax = plt.subplots(figsize=(11, 11))

# Background street edges in very light grey
edges.plot(ax=ax, linewidth=0.08, color="gainsboro", alpha=0.5)

# Plot nodes coloured by degree — use a diverging colourmap
# We cap at degree 6 to avoid outliers washing out the colour scale
nodes_plot = nodes[nodes['degree'] <= 6].copy()
scatter = ax.scatter(
    nodes_plot.geometry.x,
    nodes_plot.geometry.y,
    c=nodes_plot['degree'],
    cmap="RdYlGn",       # Red = low connectivity, Green = high connectivity
    s=0.8,               # Small dot size so you can see the pattern
    alpha=0.6,
    vmin=1, vmax=6
)

# Colourbar
cb = plt.colorbar(scatter, ax=ax, shrink=0.6, pad=0.02)
cb.set_label("Node Degree (number of connecting streets)", fontsize=9)
cb.set_ticks([1, 2, 3, 4, 5, 6])
cb.ax.set_yticklabels(["1 (dead end)", "2", "3", "4", "5", "6+ (hub)"], fontsize=8)

# Title
ax.set_title("Walking Network Node Degree — Liverpool, England",
             fontsize=16, fontweight="bold", pad=15)
ax.text(0.5, 0.97, "Green nodes = higher street connectivity | Red nodes = lower connectivity (dead ends)",
        transform=ax.transAxes, fontsize=9, ha="center", color="grey", style="italic")

# Caption
ax.text(0.01, 0.01, "Data: OpenStreetMap contributors (2024) | Analysis: OSMnx (Boeing, 2017)",
        transform=ax.transAxes, fontsize=8, color="grey")

# North arrow
ax.annotate("N ↑", xy=(0.97, 0.05), xycoords="axes fraction",
            fontsize=13, ha="center", fontweight="bold", color="#333333")

ax.set_facecolor("white")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 8. Interactive Walkability Map

The interactive map allows users to explore Liverpool's walking network more closely. It supports a more detailed view of street patterns and connectivity.

In [ ]:
from folium.plugins import HeatMap
import branca.colormap as cm

# ── Base map ──────────────────────────────────────────────────────────────────
# CartoDB Positron is a neutral light basemap — doesn't compete with our data
liverpool_map = folium.Map(
    location=[nodes.geometry.y.mean(), nodes.geometry.x.mean()],
    zoom_start=12,
    tiles="CartoDB positron"
)

# ── Layer 1: Sampled street network edges ─────────────────────────────────────
# We sample 6000 edges to keep the map loading fast in a browser
# Using random_state=42 so results are reproducible each time you run it
edges_sample = edges.sample(6000, random_state=42)
network_layer = folium.FeatureGroup(name="🚶 Walking Street Network (sample)", show=True)

for idx, row in edges_sample.iterrows():
    if row.geometry.geom_type == "LineString":
        coords = [[y, x] for x, y in row.geometry.coords]
        folium.PolyLine(
            locations=coords,
            color="#444444",
            weight=0.8,
            opacity=0.25
        ).add_to(network_layer)

network_layer.add_to(liverpool_map)

# ── Layer 2: Intersection density heatmap ────────────────────────────────────
# This shows WHERE intersections cluster — warm colours = denser network
heat_data = [[row.geometry.y, row.geometry.x] for idx, row in nodes.iterrows()]

HeatMap(
    heat_data,
    radius=6,
    blur=10,
    min_opacity=0.1,
    max_zoom=15,
    name="🔴 Intersection Density Heatmap",
    show=True
).add_to(liverpool_map)

# ── Layer 3: Node degree — coloured circle markers ───────────────────────────
# Each node is coloured by its degree (number of connecting streets)
# Green = highly connected | Red = dead end or low connectivity
# We only show nodes with degree >= 3 (actual intersections, not just path midpoints)
# and sample 3000 to keep it fast

degree_layer = folium.FeatureGroup(name="🟢 Node Degree (Connectivity)", show=False)

# Colour scale: red (degree 1) → green (degree 5+)
colormap = cm.LinearColormap(
    colors=["#d73027", "#fc8d59", "#fee090", "#91cf60", "#1a9850"],
    vmin=1, vmax=6,
    caption="Node Degree (number of connecting streets)"
)

nodes_intersections = nodes[nodes['degree'] >= 3].sample(3000, random_state=42)

for idx, row in nodes_intersections.iterrows():
    degree_val = int(row['degree'])
    colour = colormap(min(degree_val, 6))
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color=colour,
        fill=True,
        fill_color=colour,
        fill_opacity=0.7,
        weight=0,
        tooltip=f"Node degree: {degree_val} streets"
    ).add_to(degree_layer)

degree_layer.add_to(liverpool_map)
colormap.add_to(liverpool_map)

# ── Map bounds and controls ───────────────────────────────────────────────────
minx, miny, maxx, maxy = nodes.total_bounds
liverpool_map.fit_bounds([[miny, minx], [maxy, maxx]])

# Layer control lets users toggle each layer on/off
folium.LayerControl(collapsed=False).add_to(liverpool_map)

# ── Save and display ──────────────────────────────────────────────────────────
liverpool_map.save("liverpool_walkability_interactive.html")

# This is the correct way to display in Jupyter — avoids the "trust notebook" error
from IPython.display import IFrame
IFrame("liverpool_walkability_interactive.html", width="100%", height=550)

## 9. Key Findings

The walking network shows how Liverpool's urban structure supports movement across different areas. Denser and more connected areas are likely to offer more route choices for pedestrians, while areas with lower connectivity may provide fewer walking options.

This project demonstrates how OpenStreetMap data, street network analysis, and geospatial visualisation can be used to explore walkability in an urban setting.

## 10. Future Improvements

Future improvements could include adding population data, public transport stops, green spaces, slope/elevation, pedestrian crossings, or points of interest to create a more complete walkability analysis.

## References

Ewing, R. and Cervero, R. (2017) ‘Does compact development make people drive less? The answer is yes’, *Journal of the American Planning Association*, 83(1), pp. 19–25.

Frank, L.D., Sallis, J.F., Saelens, B.E., Leary, L., Cain, K., Conway, T.L. and Hess, P.M. (2010) ‘The development of a walkability index: Application to the Neighborhood Quality of Life Study’, *British Journal of Sports Medicine*, 44(13), pp. 924–933.

Haklay, M. and Weber, P. (2008) ‘OpenStreetMap: User-generated street maps’, *IEEE Pervasive Computing*, 7(4), pp. 12–18.

Marshall, W.E. and Garrick, N.W. (2010) ‘Street network types and road safety: A study of 24 California cities’, *Urban Design International*, 15(3), pp. 133–147.

OpenStreetMap contributors (2024) *OpenStreetMap*. Available at: https://www.openstreetmap.org/

Boeing, G. (2017) ‘OSMnx: New methods for acquiring, constructing, analyzing, and visualizing complex street networks’, *Computers, Environment and Urban Systems*, 65, pp. 126–139.

Southworth, M. (2005) ‘Designing the walkable city’, *Journal of Urban Planning and Development*, 131(4), pp. 246–257.

Vale, D.S., Saraiva, M. and Pereira, M. (2016) ‘Active accessibility: A review of operational measures of walking and cycling accessibility’, *Journal of Transport and Land Use*, 9(1), pp. 209–235.

Cervero, R. and Kockelman, K. (1997) 'Travel demand and the 3Ds: Density, diversity, and design', Transportation Research Part D: Transport and Environment, 2(3), pp. 199–219.

Shneiderman, B. (1996) 'The eyes have it: A task by data type taxonomy for information visualizations', in Proceedings of the 1996 IEEE Symposium on Visual Languages, pp. 336–343.